In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# =========================
# 1) LOAD RETURNS
# =========================
returns = pd.read_csv(
    "../data/processed/Returns.csv",
    index_col=0,
    parse_dates=True
).sort_index()

# =========================
# 2) PARAMETERS
# =========================
WINDOW = 120
MIN_OBS = 36
STALE_THRESHOLD = 0.5
START_YEAR = 2013
END_YEAR = 2024

# =========================
# 3) MIN VAR SOLVER
# =========================
def solve_min_variance(cov_matrix):

    n = cov_matrix.shape[0]
    Sigma = cov_matrix.values

    def objective(w):
        return w.T @ Sigma @ w

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1}
    ]

    bounds = [(0, 1)] * n
    x0 = np.ones(n) / n

    res = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if not res.success:
        raise ValueError(f"Optimization failed: {res.message}")

    return pd.Series(res.x, index=cov_matrix.index)

# =========================
# 4) STORAGE
# =========================
all_weights = []
all_portfolio_returns = []

# =========================
# 5) ROLLING + BACKTEST
# =========================
december_dates = returns.index[returns.index.month == 12]
december_dates = december_dates[
    (december_dates.year >= START_YEAR) &
    (december_dates.year <= END_YEAR)
]

for dec_date in december_dates:

    next_year = dec_date.year + 1

    # ----- WINDOW DATA
    window_data = returns.loc[:dec_date].tail(WINDOW)

    if len(window_data) < WINDOW:
        continue

    # =========================
    # CLEANING AVANT OPTIMISATION 🔥
    # =========================

    # assez de données
    window_data = window_data.loc[:, window_data.count() >= MIN_OBS]

    # enlever actifs avec trop de zéros (stale)
    zero_ratio = window_data.eq(0).sum() / window_data.count()
    window_data = window_data.loc[:, zero_ratio <= STALE_THRESHOLD]

    # enlever variance nulle
    window_data = window_data.loc[:, window_data.std() > 1e-8]

    if window_data.shape[1] < 2:
        continue

    # =========================
    # COVARIANCE
    # =========================
    cov_Y = window_data.cov() * (WINDOW - 1) / WINDOW

    # supprimer NaN
    cov_Y = cov_Y.dropna(axis=0, how='any').dropna(axis=1, how='any')

    # alignement
    cov_Y = cov_Y.loc[cov_Y.index, cov_Y.index]

    if cov_Y.shape[0] < 2:
        continue

    # 🔥 REGULARISATION (clé)
    cov_Y = cov_Y + 1e-6 * np.eye(len(cov_Y))

    # =========================
    # OPTIMISATION
    # =========================
    try:
        w0 = solve_min_variance(cov_Y)
    except:
        continue

    # stocker poids
    weights_row = pd.DataFrame({
        "RebalanceDate": [dec_date] * len(w0),
        "ISIN": w0.index,
        "Weight": w0.values
    })
    all_weights.append(weights_row)

    # =========================
    # BACKTEST
    # =========================
    next_returns = returns[returns.index.year == next_year][w0.index]

    if next_returns.empty:
        continue

    current_weights = w0.copy()

    for dt, r_t in next_returns.iterrows():

        available = r_t.notna()

        if available.sum() == 0:
            continue

        w_used = current_weights[available]
        w_used = w_used / w_used.sum()

        r_used = r_t[available]

        rp_t = (w_used * r_used).sum()

        all_portfolio_returns.append({
            "Date": dt,
            "Return": rp_t
        })

        # update weights
        updated = w_used * (1 + r_used) / (1 + rp_t)

        new_weights = pd.Series(0, index=current_weights.index)
        new_weights.loc[updated.index] = updated

        if new_weights.sum() > 0:
            new_weights = new_weights / new_weights.sum()

        current_weights = new_weights

# =========================
# 6) EXPORT
# =========================
weights_df = pd.concat(all_weights, ignore_index=True)

returns_df = pd.DataFrame(all_portfolio_returns)
returns_df["Date"] = pd.to_datetime(returns_df["Date"])
returns_df = returns_df.sort_values("Date")

returns_df["Cumulative"] = (1 + returns_df["Return"]).cumprod() - 1

weights_df.to_csv("../data/processed/mv_weights.csv", index=False)
returns_df.to_csv("../data/processed/mv_portfolio_returns.csv", index=False)

print("MVP + Backtest terminé ✅")

NameError: name 'window_data' is not defined